# Train your VeriCode adapter on a free GPU

This notebook fine-tunes Qwen2.5-Coder-1.5B-Instruct with 4-bit LoRA. It does **not** train a language model from scratch. The 144 self-authored expression repairs are a teaching dataset; useful repair improvement is unproven. No QuixBugs, frozen evaluator tests, or reference files enter training.

1. Sign in to Google Colab.
2. Select **Runtime → Change runtime type → GPU** using the free tier. If no free GPU is available, stop and try later; do not enable paid compute.
3. Run cells in order. First run a two-step smoke check, then at most three epochs with early stopping.
4. Download the adapter and manifest before the runtime expires.

Free GPU availability and duration are not guaranteed: https://research.google.com/colaboratory/faq.html
No API key, paid subscription, Drive mount, experiment tracker or model upload is needed. Only public model/code downloads occur. All cells are intentionally unexecuted in the repository; no trained result is claimed.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a free GPU runtime. Do not train this recipe on the laptop CPU."
print(torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


In [ ]:
import os, subprocess, sys
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.6", "peft==0.18.1", "bitsandbytes==0.49.1", "accelerate==1.12.0"], check=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path
repository = Path("/content/VeriCode")
repository_url = "https://github.com/Hassaan-Ahmad906/VeriCode-AI-debugging-and-repair-agent.git"
if repository.exists():
    if not (repository / ".git").exists():
        raise RuntimeError("Existing folder is not a Git checkout. Choose a different repository path; keep existing files.")
    root = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "--show-toplevel"], text=True).strip()
    origin = subprocess.check_output(["git", "-C", str(repository), "remote", "get-url", "origin"], text=True).strip()
    if Path(root).resolve() != repository.resolve() or origin.rstrip("/").removesuffix(".git").casefold() != repository_url.removesuffix(".git").casefold():
        raise RuntimeError("Existing checkout has a different origin or root. Choose a different path; keep existing files.")
    print("Reusing verified VeriCode checkout; existing files and revision are unchanged.")
else:
    subprocess.run(["git", "clone", repository_url, str(repository)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(repository)], check=True)
os.chdir(repository)
print("Training source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import json, urllib.request
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
# Resolve an immutable model revision before downloading weights; saved in the manifest.
with urllib.request.urlopen("https://huggingface.co/api/models/" + BASE_MODEL, timeout=30) as response:
    MODEL_REVISION = json.load(response)["sha"]
print("Base model:", BASE_MODEL, "revision:", MODEL_REVISION)
from vericode.training_data import build_examples, dataset_manifest
print(json.dumps(dataset_manifest(build_examples()), indent=2))


## Verify the runtime before training

The smoke run takes two optimizer steps, saves a disposable adapter, and reports validation loss. It is not a complete training result. If CUDA/bitsandbytes fails or memory is insufficient, stop and retain the error; do not silently substitute CPU or paid compute.


In [ ]:
smoke_output = "/content/vericode-smoke"
subprocess.run([sys.executable, "scripts/train_lora.py", "--base-model", BASE_MODEL,
    "--revision", MODEL_REVISION, "--output", smoke_output, "--smoke"], check=True)


## Train with bounded compute

Batch size 1, gradient accumulation 8, context at most 1024, rank 8 on q/v projections, maximum 3 epochs, and early stopping after a non-improving validation epoch. More epochs can overfit this small dataset; maximum training time is not the same as maximum useful learning. The prompt is masked from the training loss. Long examples fail instead of being truncated silently.


In [ ]:
training_output = "/content/vericode-trained"
subprocess.run([sys.executable, "scripts/train_lora.py", "--base-model", BASE_MODEL,
    "--revision", MODEL_REVISION, "--output", training_output, "--epochs", "3"], check=True)
manifest = json.loads((Path(training_output) / "training_manifest.json").read_text())
print(json.dumps(manifest, indent=2))


## Save your adapter

This archive contains LoRA weights, tokenizer configuration, source/model revisions, package versions, dataset hashes and observed training/validation losses. It contains no base-model weights. Loss changes alone do not establish repair correctness. Use independent Docker execution to compare the base model and adapter before claiming improvement.

A Qwen safetensors adapter cannot be assumed to import directly into every Ollama release. The companion training guide explains merging and GGUF conversion. The laptop UI works immediately with the pretrained local model; replacing it with a validated trained export is a separate step.


In [ ]:
import shutil
archive = shutil.make_archive("/content/vericode-adapter", "zip", training_output)
from google.colab import files
files.download(archive)


## Optional: export your trained model for the laptop

Run this only after the complete training run succeeded. It merges the adapter with the exact base, preserves the base license, builds the pinned official llama.cpp CPU quantizer, and converts to Q4_K_M GGUF. Conversion is a separate, unexecuted recipe in this repository; retain errors if the cloud environment is incompatible. It needs several GB of temporary cloud disk and downloads a roughly 1 GB final model to the laptop. No paid service is used. The download can take time.


In [ ]:
subprocess.run([sys.executable, "scripts/merge_adapter.py", "--training-dir", training_output,
    "--output", "/content/vericode-merged"], check=True)
conversion_root = Path("/content/llama.cpp")
CONVERSION_REVISION = "2a667032dc788c3a071d538e0b572e45a3f12731"
if conversion_root.exists():
    raise FileExistsError("Preserve existing converter checkout; use a fresh path.")
subprocess.run(["git", "clone", "--no-checkout", "https://github.com/ggml-org/llama.cpp.git", str(conversion_root)], check=True)
subprocess.run(["git", "-C", str(conversion_root), "checkout", CONVERSION_REVISION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
    str(conversion_root / "requirements/requirements-convert_hf_to_gguf.txt")], check=True)
subprocess.run(["cmake", "-S", str(conversion_root), "-B", str(conversion_root / "build"),
    "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF"], check=True)
subprocess.run(["cmake", "--build", str(conversion_root / "build"), "--target", "llama-quantize", "-j", "2"], check=True)
subprocess.run([sys.executable, str(conversion_root / "convert_hf_to_gguf.py"), "/content/vericode-merged",
    "--outfile", "/content/vericode-f16.gguf", "--outtype", "f16"], check=True)
subprocess.run([str(conversion_root / "build/bin/llama-quantize"),
    "/content/vericode-f16.gguf", "/content/vericode-q4.gguf", "Q4_K_M"], check=True)
import hashlib
digest = hashlib.sha256()
with open("/content/vericode-q4.gguf", "rb") as stream:
    while block := stream.read(1024 * 1024):
        digest.update(block)
export_manifest = {"training": manifest, "converter_revision": CONVERSION_REVISION,
    "quantization": "Q4_K_M", "gguf_sha256": digest.hexdigest(),
    "claims": "Exported fine-tuned weights; repair improvement remains unmeasured."}
Path("/content/vericode-export.json").write_text(json.dumps(export_manifest, indent=2))
files.download("/content/vericode-q4.gguf")
files.download("/content/vericode-export.json")
files.download("/content/vericode-merged/LICENSE")


## Import on the laptop

Copy the downloaded GGUF, export JSON and LICENSE into an ignored folder such as `artifacts/trained-model`. Create a `Modelfile` there containing `FROM ./vericode-q4.gguf` and `PARAMETER num_ctx 4096` on separate lines. With the local Ollama server running, use:

```powershell
.\\artifacts\\local-model\\runtime\\ollama.exe create vericode-custom -f artifacts/trained-model/Modelfile
```

In VeriCode choose provider **ollama** and model tag **vericode-custom**. Run Docker visible checks and independent evaluation where available. Compare against the original base model before making quality claims. Renamed examples and lower training loss do not establish generalization.
